## 文献调研SurveyAgent案例

本案例演示如何单独使用 `SurveyAgent` 对给定研究主题做多源文献检索与筛选。

### 1. 环境设置与模块导入

In [7]:
import json
import os
import sys

sys.path.append("../../")

from vibescience_agent.config import VibeScienceConfig
from vibescience_agent.agents.agent_factory import AgentFactory
from vibescience_agent.model.model_factory import ModelFactory

### 2. Semantic Scholar API 密钥（环境变量 `S2_API_KEY`）

当 `vibescience.yaml` 中 `tools.paper_survey.sources` 包含 `semantic_scholar` 时，底层会读取环境变量 **`S2_API_KEY`**。

推荐做法：在操作系统或终端中预先 `set` / `export S2_API_KEY=...`。若未设置，下面单元会使用 Notebook 内提供的默认值。

In [ ]:
# 若已在环境中配置 S2_API_KEY，则不会覆盖；否则使用下列默认值
os.environ.setdefault(
    "S2_API_KEY",
    "***",  # 用户自己的真实的密钥
)
print("S2_API_KEY configured:", bool(os.environ.get("S2_API_KEY")))

S2_API_KEY configured: True


### 3. 加载配置文件

加载实验配置文件，可以根据实际情况修改，**配置文件中需包含用户自己的模型 API 密钥等信息**。

文献工具由 **`tools.paper_survey`** 控制（`sources`、`max_results`）。本目录配置默认包含 **PubMed、arXiv、Semantic Scholar** 三源。

In [9]:
config_path = "./vibescience.yaml"
config = VibeScienceConfig.init_config_from_yaml(config_path)

[INFO] 2026-04-09-11:14:51.108.000 [vibescience_agent\config\vibescience_config.py:309] 
VIBESCIENCE CONFIGURATION
├─ version: 1.0.0
├─ agents:
│   ├─ survey:
│   │   ├─ agent_type: survey
│   │   ├─ model_config:
│   │   │   ├─ model_name: kimi-k2.5
│   │   │   ├─ api_key: ***
│   │   │   ├─ base_url: https://dashscope.aliyuncs.com/compatible-mode/v1
│   │   │   ├─ provider: openai
│   │   │   ├─ temperature: 0.2
│   │   │   ├─ max_tokens: 4096
│   │   │   ├─ timeout: 60
│   │   │   ├─ max_retries: 2
│   │   │   └─ max_connections: 8
│   │   ├─ max_retries: 2
│   │   ├─ use_tool_retriever: False
│   │   ├─ skill_path: 
│   │   └─ max_papers: 5
│   ├─ plan:
│   │   ├─ agent_type: plan
│   │   ├─ model_config:
│   │   │   ├─ model_name: your-model-id
│   │   │   ├─ api_key: ***
│   │   │   ├─ base_url: https://your-api-endpoint.example/v1
│   │   │   ├─ provider: openai
│   │   │   ├─ temperature: 0.7
│   │   │   ├─ max_tokens: 4096
│   │   │   ├─ timeout: 60
│   │   │   ├─ max_retries:

### 4. 初始化 SurveyAgent

通过 `AgentFactory` 创建 `SurveyAgent`，并注入 `ModelFactory` 生成的模型实例。`tool_config` 中的 `paper_survey` 与测试中 `_create_tool_config()` 一致，由 YAML 的 `tools` 段提供。

In [10]:
model_factory = ModelFactory()
survey_agent = AgentFactory.create_agent(
    agent_type="survey",
    config=config.get_agent_config("survey"),
    tool_config=config.tools,
    model_factory=model_factory,
)

survey_cfg = config.get_agent_config("survey")
print(
    "SurveyAgent ready:",
    "max_papers=", getattr(survey_cfg, "max_papers", None),
    "model=", survey_cfg.model_config.model_name,
)

[INFO] 2026-04-09-11:14:51.115.000 [vibescience_agent\agents\agent_factory.py:79] Created agent instance: survey (SurveyAgent)


SurveyAgent ready: max_papers= 5 model= kimi-k2.5


### 5. 定义调研问题

下面为**通用生物医学信息学 / 医疗 AI** 主题示例。

In [15]:
query = (
    "Survey recent machine learning methods for healthcare risk prediction and clinical decision support, "
    "with emphasis on interpretability and validation benchmarks; prioritize work from 2023–2025."
)
messages = [{"role": "user", "content": query}]

print("User query (first message content):\n", query)

User query (first message content):
 Survey recent machine learning methods for healthcare risk prediction and clinical decision support, with emphasis on interpretability and validation benchmarks; prioritize work from 2023–2025.


### 6. 执行文献调研并展示结果

调用 `await survey_agent.execute(messages)`。该过程包含多轮模型调用与外部检索，**可能耗时数分钟**。返回值为论文字典列表；下面打印篇数与前若干条的标题、来源与分数。

In [12]:
papers = await survey_agent.execute(messages)

print(f"Returned {len(papers)} papers\n")
for i, p in enumerate(papers[:10]):
    title = p.get("title", "")
    src = p.get("source", "")
    score = p.get("score", "")
    suf = "..." if len(title) > 120 else ""
    print(f"{i + 1}. [{src}] score={score} {title[:120]}{suf}")

[ERROR] 2026-04-09-11:15:00.273.000 [vibescience_agent\tools\paper_survey.py:158] Error searching PubMed: ('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None))
c:\Users\zhentao\.conda\envs\msAgent\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\zhentao\.conda\envs\msAgent\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Returned 5 papers

1. [arXiv] score=8 Chain-of-Thought Prompting Elicits Reasoning in Large Language Models
2. [arXiv] score=8 DiffCoT: Diffusion-styled Chain-of-Thought Reasoning in LLMs
3. [arXiv] score=7 Understanding Reasoning in Chain-of-Thought from the Hopfieldian View
4. [arXiv] score=7 Making Large Language Models Better Reasoners with Alignment
5. [arXiv] score=7 Large Language Models Reasoning Abilities Under Non-Ideal Conditions After RL-Fine-Tuning


### 7. 查看单条结果结构

In [13]:
if papers:
    print(json.dumps(papers[0], ensure_ascii=False, indent=2))
else:
    print("No papers returned.")

{
  "id": "0",
  "title": "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models",
  "authors": [
    "Jason Wei",
    "Xuezhi Wang",
    "Dale Schuurmans",
    "Maarten Bosma",
    "Brian Ichter",
    "Fei Xia",
    "Ed Chi",
    "Quoc Le",
    "Denny Zhou"
  ],
  "abstract": "We explore how generating a chain of thought -- a series of intermediate reasoning steps -- significantly improves the ability of large language models to perform complex reasoning. In particular, we show how such reasoning abilities emerge naturally in sufficiently large language models via a simple method called chain of thought prompting, where a few chain of thought demonstrations are provided as exemplars in prompting. Experiments on three large language models show that chain of thought prompting improves performance on a range of arithmetic, commonsense, and symbolic reasoning tasks. The empirical gains can be striking. For instance, prompting a 540B-parameter language model with just eigh